# Single-Gamma Normalization on MDA-MB-468 Cisplatin CyTOF Dataset

This notebook performs end-to-end data ingestion, quality control, single-gamma normalization, z-scoring, UMAP embedding, and Leiden clustering for the **MDA-MB-468 Cisplatin** single-cell CyTOF dataset:
- `c01_UT_all_mod.csv` (Untreated)
- `c02_OT_all_mod.csv` (Off-Treatment)
- `c03_T_all_mod.csv` (Treated)
- `c04_T_noDeadRemove_all_mod.csv` (Treated without dead cell removal)

**Analysis Structure:**
1. **Individual Sample Analysis (`c01_UT`)**: Per-marker z-scoring computed independently within `c01_UT`, UMAP embeddings, all-marker UMAP grids, and cluster heatmaps for both feature sets.
2. **Joint Combined Analysis**: Group-balanced z-scoring (`zscore_markers_balanced` grouped by `sample_id`) computed across the combined dataset.

**Core Histones & Marker Configuration:**
- **Control / Core Histones**: `['H3.3', 'H4']` only.
- **Removed Markers**: `H3`, `H2A`, and `DNA` are completely excluded from analysis and visualization.
- **Corrected Markers**: Includes `E-cadherin` alongside all remaining intracellular markers.

**Feature Sets Compared:**
- **Set 1 (`EPI`)**: All 18 histone PTMs in the panel (with `H3K27me3` and `H3K36me3`).
- **Set 2 (`EPI_SUB`)**: 16 histone PTMs excluding `H3K27me3` and `H3K36me3`.


## 0. Setup

In [ ]:
import sys, shutil, warnings, json
from pathlib import Path

# cytof_transform is loaded via sys.path injection
CT_PATH = "/Users/ronguy/Dropbox/Work/CyTOF/Code/cytof-transform"
if CT_PATH not in sys.path:
    sys.path.insert(0, CT_PATH)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as ss

import scanpy as sc

import cytof_transform
from cytofstandard import Project

warnings.filterwarnings("ignore", category=UserWarning, module="umap")
sc.settings.verbosity = 1
%matplotlib inline

plt.rcParams.update({
    'axes.labelsize': 11, 'axes.titlesize': 11,
    'xtick.labelsize': 9, 'ytick.labelsize': 9,
    'figure.dpi': 110, 'pdf.fonttype': 42, 'ps.fonttype': 42,
})
sns.set_style("white")

print("cytof_transform", cytof_transform.__version__, "->", cytof_transform.__file__)
try:
    import CyTOFHelper  # noqa: F401
    print("!! WARNING: CyTOFHelper is importable — the divide path will use the LEGACY implementation.")
except ModuleNotFoundError:
    print("CyTOFHelper not importable -> native std-min divide will be used (intended).")


In [ ]:
def uns_history(adata, key):
    """Read uns[key]['history'] as a list of dicts safely."""
    node = adata.uns.get(key, None)
    if node is None or not hasattr(node, "get"):
        return []
    hist = node.get("history", None)
    if hist is None:
        return []
    out = []
    for e in list(hist):
        if isinstance(e, (str, bytes)):
            try:
                e = json.loads(e)
            except Exception:
                continue
        if isinstance(e, dict):
            out.append(e)
    return out


In [ ]:
LINE       = "MDAMB468_Cisplatin"
LINE_DISP  = "MDA-MB-468 Cisplatin"

BASE       = Path("/Users/ronguy/Dropbox/Work/CyTOF/CyTOF_Christina")
DATA_DIR   = Path("/Users/ronguy/Dropbox/CyTOF_Breast/CyTOF_CR7/072026_for_Guy/HadasSangitaMDAMB468_Cisplatin")
PLOTS      = BASE / "Plots"; PLOTS.mkdir(exist_ok=True)

REG          = Path("/Users/ronguy/Dropbox/Work/CyTOF/Code/CyTOFSTD/cytof_marker_registry_files")
PROJECT_PATH = Path(f"/Users/ronguy/Dropbox/Work/CyTOF/Projects/{LINE}_SingleGamma")
RUN_ID       = "MDAMB468_Cisplatin"


In [ ]:
ARCSINH_COFACTOR = 5.0
SEED             = 42
REBUILD          = False   # Set True to rebuild project from scratch

METHODS = {
    "single": ("regress, single γ", "#33a02c", dict(method="regress", gamma_mode="single")),
}
MKEYS  = list(METHODS)
LABEL  = {k: v[0] for k, v in METHODS.items()}
COLOR  = {k: v[1] for k, v in METHODS.items()}

LAYER_OF = {k: f"norm_{k}"    for k in MKEYS}   # corrected, arcsinh scale
ZLAYER_OF= {k: f"norm_{k}_z"  for k in MKEYS}   # z from cytof_transform
ZS_OF    = {k: f"norm_{k}_zs" for k in MKEYS}   # explicit balanced z-score step
TF_OF    = {k: f"tf_{k}"     for k in MKEYS}

EMB_ALL  = "umap_single"
CLKEY_ALL= "cl_single"

EMB_SUB  = "umap_single_sub"
CLKEY_SUB= "cl_single_sub"


## 1. Initialize CyTOFSTD Project

In [ ]:
if REBUILD and PROJECT_PATH.exists():
    shutil.rmtree(PROJECT_PATH)

try:
    project = Project.load(str(PROJECT_PATH))
    print(f"Loaded existing project at {PROJECT_PATH}")
except Exception:
    PROJECT_PATH.parent.mkdir(parents=True, exist_ok=True)
    project = Project.create(
        str(PROJECT_PATH),
        project_id=f"{LINE}_SingleGamma",
        project_name=f"{LINE_DISP} single-gamma normalization",
        standard_marker_file=str(REG / "standard_markers.csv"),
        marker_alias_file=str(REG / "marker_aliases.yaml"),
    )
    print(f"Created project at {PROJECT_PATH}")


## 2. Preprocess Raw Files and Ingest

In [ ]:
PREP_DIR = PROJECT_PATH / "preprocessed"
PREP_DIR.mkdir(parents=True, exist_ok=True)

raw_csv_files = sorted(DATA_DIR.glob("*.csv"))
prep_files = []
sample_meta = []

for f in raw_csv_files:
    df = pd.read_csv(f)
    # Strip whitespace from column names (e.g. ' Vimentin' -> 'Vimentin')
    df.columns = [c.strip() for c in df.columns]
    
    # Map marker alias 'H2A119ub' -> 'H2AK119ub' for standard registry alignment
    if 'H2A119ub' in df.columns:
        df.rename(columns={'H2A119ub': 'H2AK119ub'}, inplace=True)
        
    out_path = PREP_DIR / f.name
    df.to_csv(out_path, index=False)
    prep_files.append(out_path)
    
    sample_id = f.stem.replace("_all_mod", "")
    cond = sample_id.split("_")[1] if "_" in sample_id else sample_id
    sample_meta.append({
        "file_name": f.name,
        "sample_id": sample_id,
        "line_id":   LINE,
        "condition": cond,
    })

meta_path = PREP_DIR / "samples.csv"
pd.DataFrame(sample_meta).to_csv(meta_path, index=False)

run_ingested = False
if project.has_run(RUN_ID):
    run = project.get_run(RUN_ID)
    try:
        adata = run.read_adata()
        run_ingested = True
        print(f"Run '{RUN_ID}' already ingested — skipping ingestion.")
    except Exception:
        print(f"Run '{RUN_ID}' registered but not ingested — ingesting now...")
else:
    run = project.add_run(RUN_ID, run_name=f"{LINE_DISP}")

if not run_ingested:
    run.ingest(
        files=[str(f) for f in prep_files],
        sample_metadata=str(meta_path),
        strict_markers=False,
        allow_extra_markers=True,
        show_marker_coverage=True,
    )
    print("Ingested successfully.")

adata = run.read_adata()
print(f"\nAnnData: {adata.n_obs:,} cells x {adata.n_vars} markers across {adata.obs['sample_id'].nunique()} samples")
print(adata.obs['sample_id'].value_counts())


In [ ]:
run.ingestion_summary()

## 3. Quality Control — Core Histones (H3.3 & H4) > 5
Filter out debris/low-signal events: core histones `H3.3 > 5` and `H4 > 5` on raw counts.


In [ ]:
CORE_HISTONES = ["H3.3", "H4"]  # User instruction: use H3.3 and H4 only
print("Core histones (control markers):", CORE_HISTONES)

pre = run.read_adata()
raw_pre = pd.DataFrame(pre.layers["raw"], columns=pre.var_names, index=pre.obs_names)

for m in CORE_HISTONES:
    n_bad = int((raw_pre[m] <= 5).sum())
    print(f"  {m:6s}  <=5 : {n_bad:5d} cells ({100*n_bad/len(raw_pre):.2f}%)")


In [ ]:
fig, axes = plt.subplots(1, len(CORE_HISTONES), figsize=(4.2 * len(CORE_HISTONES), 3.2))
for ax, m in zip(np.atleast_1d(axes), CORE_HISTONES):
    ax.hist(np.log10(raw_pre[m] + 1), bins=200, color="0.35")
    ax.axvline(np.log10(6), color="crimson", lw=1.5, ls="--", label="gate (>5)")
    ax.set_title(m); ax.set_xlabel("log10(raw + 1)"); ax.set_yscale("log")
    ax.legend(fontsize=8)
axes[0].set_ylabel("cells")
fig.suptitle("QC gate on core histones H3.3 & H4 (raw scale)", y=1.03)
plt.tight_layout(); plt.show()


In [ ]:
qc_hist = uns_history(run.read_adata(), "qc")
if qc_hist:
    h = qc_hist[-1]
    print(f"QC already applied — skipping to stay idempotent.")
    print(f"  n_before={h['n_before']:,}  n_after={h['n_after']:,}  n_removed={h['n_removed']:,}")
else:
    mask = run.qc_gate({m: {"lower": 5} for m in CORE_HISTONES}, layer="raw", inplace=True)
    print(f"QC: kept {int(mask.sum()):,} / {len(mask):,} cells (removed {int((~mask).sum()):,}, {100*(~mask).mean():.2f}%)")

adata = run.read_adata()
print(f"\nPost-QC AnnData: {adata.n_obs:,} cells x {adata.n_vars} markers")


## 4. Marker Categorization & Feature Sets

We define two epigenetic feature sets for comparative UMAP embedding & clustering:
- **Core Histones**: `['H3.3', 'H4']` (used for technical factor extraction).
- **Excluded Markers**: `H3`, `H2A`, and `DNA` are completely removed per instruction.
- **Corrected Markers (`TO_CORRECT`)**: All non-core histone intracellular markers **plus `E-cadherin`**.
- **Set 1 (`EPI`)**: All 18 epigenetic modifications present in the panel (with `H3K27me3` & `H3K36me3`).
- **Set 2 (`EPI_SUB`)**: 16 epigenetic modifications, **excluding `H3K27me3` and `H3K36me3`**.


In [ ]:
# Exclude H3, H2A, and DNA from all downstream correction, analysis, and plotting
IGNORE_MARKERS = ['H3', 'H2A', 'DNA']
ALL_MARKERS = [m for m in run.read_adata().var_names.tolist() if m not in IGNORE_MARKERS]

# Core histones: H3.3 and H4 only
CORE_HISTONES = ["H3.3", "H4"]

# Epigenetic modifications: Set 1 (All EPI) & Set 2 (EPI w/o H3K27me3 & H3K36me3)
EPI_CANDIDATES = ['H2AK119ub','H2BK5ac','H3K27ac','H3K27me2','H3K27me3','H3K36me2','H3K36me3',
                  'H3K4me1','H3K4me2','H3K4me3','H3K64ac','H3K9ac','H3K9me2','H3K9me3',
                  'H3S28p','H4K16ac','H4K20me3','pH2A.X']
EPI     = [m for m in EPI_CANDIDATES if m in ALL_MARKERS]
EXCLUDE = ['H3K27me3', 'H3K36me3']
EPI_SUB = [m for m in EPI if m not in EXCLUDE]

# Surface markers (extracellular) — E-cadherin is explicitly moved to corrected markers
SURFACE = [m for m in run.markers_extracellular if m in ALL_MARKERS and m.lower() not in ['e-cadherin', 'ecadherin']]

# Intracellular & markers to correct (including E-cadherin, excluding core histones and ignored markers)
INTRA        = [m for m in ALL_MARKERS if m not in SURFACE and m not in CORE_HISTONES]
TO_CORRECT   = [m for m in INTRA if m not in CORE_HISTONES]
INTRA_OTHER  = [m for m in TO_CORRECT if m not in EPI]

MARKERS_POST = [m for m in ALL_MARKERS if m not in CORE_HISTONES]
OTHER        = [m for m in MARKERS_POST if m not in EPI]

print(f"IGNORED MARKERS (removed)              : {IGNORE_MARKERS}")
print(f"CORE_HISTONES ({len(CORE_HISTONES):2d}) control markers   : {CORE_HISTONES}")
print(f"TO_CORRECT    ({len(TO_CORRECT):2d}) corrected          : {TO_CORRECT}")
print(f"  EPI (Set 1) ({len(EPI):2d}) -> UMAP/clustering : {EPI}")
print(f"  EPI_SUB (Set 2) ({len(EPI_SUB):2d}) (w/o H3K27me3 & H3K36me3): {EPI_SUB}")
print(f"  EXCLUDED    ({len(EXCLUDE):2d})                    : {EXCLUDE}")
print(f"  INTRA_OTHER ({len(INTRA_OTHER):2d})                    : {INTRA_OTHER}")
print(f"SURFACE       ({len(SURFACE):2d}) NOT corrected      : {SURFACE}")


## 5. Single-Gamma Normalization
Run additive PC1 regression normalization with a single shared slope $\bar{\gamma}$ (`gamma_mode="single"`) anchored at median technical factor.


In [ ]:
norm_hist  = uns_history(run.read_adata(), "normalization")
done_layers = {h.get("corrected_layer") for h in norm_hist}
print("Normalized layers already present:", done_layers or "none")


In [ ]:
summaries = {}
for k in MKEYS:
    layer = LAYER_OF[k]
    if layer in done_layers:
        summaries[k] = [h for h in norm_hist if h.get("corrected_layer") == layer][-1]
        print(f"{LABEL[k]:24s} : already applied — skipping.")
        continue
    print(f"{LABEL[k]:24s} : running ...")
    summaries[k] = run.normalize_with_cytof_transform(
        control_markers=CORE_HISTONES,
        markers_to_correct=TO_CORRECT,
        source_layer="raw",
        input_is_arcsinh=False,
        arcsinh_cofactor=ARCSINH_COFACTOR,
        groupby_col="sample_id",
        corrected_layer=layer,
        z_layer=ZLAYER_OF[k],
        anchor_to_median=True,
        zscore=True,
        **METHODS[k][2],
    )
    a = run.read_adata()
    a.obs[TF_OF[k]] = a.obs["norm_tech_factor"].values
    run.save(a)

pd.DataFrame({k: {"method": s["method"], "gamma_mode": s["gamma_mode"],
                  "tech_factor": s["tech_factor_kind"], "layer": s["corrected_layer"]}
              for k, s in summaries.items()}).T


In [ ]:
adata = run.read_adata()
raw_all   = pd.DataFrame(adata.layers["raw"], columns=adata.var_names, index=adata.obs_names)
asinh_raw = np.arcsinh(raw_all / ARCSINH_COFACTOR)

print("Layers:", sorted(adata.layers.keys()))
print()
print(f"{'':26s} {'min':>8s} {'max':>8s} {'mean':>8s}")
print(f"{'raw (arcsinh)':26s} {asinh_raw[MARKERS_POST].values.min():8.3f} "
      f"{asinh_raw[MARKERS_POST].values.max():8.3f} {asinh_raw[MARKERS_POST].values.mean():8.3f}")
for k in MKEYS:
    M = pd.DataFrame(adata.layers[LAYER_OF[k]], columns=adata.var_names, index=adata.obs_names)[MARKERS_POST].values
    print(f"{LABEL[k]:26s} {M.min():8.3f} {M.max():8.3f} {M.mean():8.3f}")


## 6. Individual Sample Analysis — Untreated (`c01_UT`)

We begin by focusing on the **Untreated sample (`c01_UT`)**:
- Perform **sample-specific z-scoring** directly within `c01_UT`.
- Compute UMAP embeddings and Leiden clustering for:
  - **Set 1 (All EPI)**: With `H3K27me3` & `H3K36me3` -> `umap_epi`, `cl_epi`.
  - **Set 2 (Sub EPI)**: Without `H3K27me3` & `H3K36me3` -> `umap_epi_sub`, `cl_epi_sub`.
- Plot UMAP cluster comparisons.
- Render **all-marker UMAP grids** on both UMAP embeddings.
- Generate **cluster heatmaps** for each of the clusterings.


In [ ]:
TARGET_SAMPLE = "c01_UT"
print(f"==========================================")
print(f"  Individual Sample Analysis: {TARGET_SAMPLE}")
print(f"==========================================")

adata = run.read_adata()
sub = adata[adata.obs["sample_id"] == TARGET_SAMPLE].copy()

# Per-sample z-scoring on normalized layer
X_norm = sub.layers[LAYER_OF['single']]
sub.layers["norm_sample_zs"] = (X_norm - X_norm.mean(axis=0)) / (X_norm.std(axis=0) + 1e-8)

# 1. Set 1: All 18 EPI (with H3K27me3 & H3K36me3)
epi_idx = [sub.var_names.get_loc(m) for m in EPI]
sub.obsm["X_epi"] = sub.layers["norm_sample_zs"][:, epi_idx]
sc.pp.neighbors(sub, n_neighbors=15, use_rep="X_epi")
sc.tl.umap(sub, min_dist=0.1, random_state=SEED)
sub.obsm["umap_epi"] = sub.obsm["X_umap"].copy()
sc.tl.leiden(sub, key_added="cl_epi", resolution=0.15, random_state=SEED)

# 2. Set 2: Sub EPI (without H3K27me3 & H3K36me3)
sub_idx = [sub.var_names.get_loc(m) for m in EPI_SUB]
sub.obsm["X_epi_sub"] = sub.layers["norm_sample_zs"][:, sub_idx]
sc.pp.neighbors(sub, n_neighbors=15, use_rep="X_epi_sub")
sc.tl.umap(sub, min_dist=0.1, random_state=SEED)
sub.obsm["umap_epi_sub"] = sub.obsm["X_umap"].copy()
sc.tl.leiden(sub, key_added="cl_epi_sub", resolution=0.15, random_state=SEED)

sub.obs["cl_epi"] = sub.obs["cl_epi"].astype(str).astype("category")
sub.obs["cl_epi_sub"] = sub.obs["cl_epi_sub"].astype(str).astype("category")

print(f"[{TARGET_SAMPLE}] Set 1 (All EPI, with H3K27me3 & H3K36me3)     -> {sub.obs['cl_epi'].nunique()} Leiden clusters")
print(f"[{TARGET_SAMPLE}] Set 2 (Sub EPI, without H3K27me3 & H3K36me3) -> {sub.obs['cl_epi_sub'].nunique()} Leiden clusters")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

# Left: Set 1 (All 18 EPI)
sc.pl.embedding(
    sub, basis="umap_epi", color="cl_epi", ax=axes[0], show=False,
    legend_loc="on data", legend_fontsize=9, legend_fontoutline=2,
    palette="tab20", size=4, frameon=False, use_raw=False,
    title=f"{TARGET_SAMPLE} — Set 1 (All 18 EPI) [{sub.obs['cl_epi'].nunique()} clusters]",
)

# Right: Set 2 (Sub EPI w/o H3K27me3 & H3K36me3)
sc.pl.embedding(
    sub, basis="umap_epi_sub", color="cl_epi_sub", ax=axes[1], show=False,
    legend_loc="on data", legend_fontsize=9, legend_fontoutline=2,
    palette="tab20", size=4, frameon=False, use_raw=False,
    title=f"{TARGET_SAMPLE} — Set 2 (Sub EPI w/o H3K27me3 & H3K36me3) [{sub.obs['cl_epi_sub'].nunique()} clusters]",
)

plt.suptitle(f"MDA-MB-468 Cisplatin — {TARGET_SAMPLE} UMAP & Leiden Clustering", y=1.02, fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(PLOTS / f"{LINE}_umap_persample_{TARGET_SAMPLE}.png", dpi=180, bbox_inches="tight")
plt.show()


In [ ]:
def sample_marker_grid(sub_obj, emb_key, markers, title_prefix, ncols=5, fname=None):
    """UMAP coloured by each z-scored marker for an individual sample."""
    nrows = int(np.ceil(len(markers) / ncols))
    fig, axes_arr = plt.subplots(nrows, ncols, figsize=(ncols * 3.8, nrows * 3.6))
    axes_flat = np.atleast_1d(axes_arr).ravel()

    for idx, mname in enumerate(markers):
        ax = axes_flat[idx]
        sc.pl.embedding(
            sub_obj, basis=emb_key, color=mname, layer="norm_sample_zs",
            cmap="seismic", vcenter=0, vmin="p1", vmax="p99",
            ax=ax, size=5, frameon=False, use_raw=False,
            colorbar_loc="right", show=False
        )
        ax.set_title(mname, fontsize=13, fontweight="bold", pad=6)

    for ax in axes_flat[len(markers):]:
        ax.axis("off")

    fig.suptitle(f"{LINE_DISP} — {TARGET_SAMPLE} — {title_prefix} — All Markers",
                 fontsize=18, fontweight="bold", y=1.02)
    plt.tight_layout()
    if fname:
        fig.savefig(PLOTS / fname, dpi=200, bbox_inches="tight")
    plt.show()

ALL_MARKERS_SORTED = sorted(EPI + OTHER, key=lambda s: s.lower())

# Marker Grid for c01_UT on Set 1 UMAP (umap_epi)
sample_marker_grid(sub, "umap_epi", ALL_MARKERS_SORTED, title_prefix="Set 1 UMAP (All EPI)", fname=f"{LINE}_{TARGET_SAMPLE}_umap_all_epi_allmarkers.png")

# Marker Grid for c01_UT on Set 2 UMAP (umap_epi_sub)
sample_marker_grid(sub, "umap_epi_sub", ALL_MARKERS_SORTED, title_prefix="Set 2 UMAP (Sub EPI w/o H3K27me3 & H3K36me3)", fname=f"{LINE}_{TARGET_SAMPLE}_umap_sub_epi_allmarkers.png")


In [ ]:
def plot_sample_cluster_heatmap(sub_obj, cluster_key, marker_list, title, fname=None):
    """Plot mean marker intensity heatmap per cluster for an individual sample."""
    df = pd.DataFrame(sub_obj.layers["norm_sample_zs"], columns=sub_obj.var_names, index=sub_obj.obs_names)
    df["cluster"] = sub_obj.obs[cluster_key].values
    cluster_means = df.groupby("cluster")[marker_list].mean()
    
    plt.figure(figsize=(10, max(3, 0.6 * len(cluster_means))))
    sns.heatmap(cluster_means, cmap="coolwarm", center=0, annot=True, fmt=".2f", cbar_kws={"label": "Z-score"})
    plt.title(title, fontsize=12, fontweight="bold", pad=10)
    plt.xlabel("Markers")
    plt.ylabel("Leiden Cluster")
    plt.tight_layout()
    if fname:
        plt.savefig(PLOTS / fname, dpi=180, bbox_inches="tight")
    plt.show()

NONEPI = [m for m in OTHER if m in sub.var_names]

# 1. Heatmap for Set 1 Clustering (cl_epi) — Epigenetic Marks
plot_sample_cluster_heatmap(sub, "cl_epi", EPI, title=f"{TARGET_SAMPLE} — Set 1 (All EPI) Clustering — Epigenetic Marks", fname=f"{LINE}_{TARGET_SAMPLE}_heatmap_set1_epi.png")

# 2. Heatmap for Set 1 Clustering (cl_epi) — Other Markers
plot_sample_cluster_heatmap(sub, "cl_epi", NONEPI, title=f"{TARGET_SAMPLE} — Set 1 (All EPI) Clustering — Non-Epigenetic Markers", fname=f"{LINE}_{TARGET_SAMPLE}_heatmap_set1_other.png")

# 3. Heatmap for Set 2 Clustering (cl_epi_sub) — Epigenetic Marks (w/o H3K27me3 & H3K36me3)
plot_sample_cluster_heatmap(sub, "cl_epi_sub", EPI_SUB, title=f"{TARGET_SAMPLE} — Set 2 (Sub EPI) Clustering — Epigenetic Marks", fname=f"{LINE}_{TARGET_SAMPLE}_heatmap_set2_epi.png")

# 4. Heatmap for Set 2 Clustering (cl_epi_sub) — Other Markers
plot_sample_cluster_heatmap(sub, "cl_epi_sub", NONEPI, title=f"{TARGET_SAMPLE} — Set 2 (Sub EPI) Clustering — Non-Epigenetic Markers", fname=f"{LINE}_{TARGET_SAMPLE}_heatmap_set2_other.png")


## 7. Joint Combined Analysis (Group-Balanced Z-Scoring)

For the combined analysis across all samples together:
- We compute **Group-Balanced Z-Scoring** (`zscore_markers_balanced` grouped by `sample_id`) to ensure each sample contributes equally to the marker mean and scaling across the entire dataset.
- We then run 2D UMAP embedding and Leiden clustering across all samples together for **Set 1 (`EPI`)** and **Set 2 (`EPI_SUB`)**.


In [ ]:
adata = run.read_adata()
for k in MKEYS:
    if ZS_OF[k] in adata.layers:
        print(f"{LABEL[k]:24s} : '{ZS_OF[k]}' exists — skipping.")
        continue
    run.zscore_markers_balanced(
        source_layer=LAYER_OF[k],
        output_layer=ZS_OF[k],
        groupby_col="sample_id",
        random_state=SEED,
    )
    print(f"{LABEL[k]:24s} : z-scored {LAYER_OF[k]} -> {ZS_OF[k]}")

adata = run.read_adata()
chk = pd.DataFrame({
    LABEL[k]: pd.DataFrame(adata.layers[ZS_OF[k]], columns=adata.var_names)[EPI]
              .agg(['mean', 'std']).T.stack()
    for k in MKEYS
}).round(4)
print("\nPer-marker mean/std of epigenetic marks after group-balanced z-scoring (should be ~0 / ~1):")
chk.groupby(level=1).agg(['min', 'max']).round(4)


In [ ]:
UMAP_KW = dict(n_neighbors=15, min_dist=0.1, metric="euclidean", random_state=SEED)
RESOLUTION = 0.15

existing_emb = set((run.read_adata().uns.get("embeddings", {}) or {}).keys())

# --- Set 1: All 18 Epigenetic Marks ---
if EMB_ALL in existing_emb:
    print(f"All EPI ('{EMB_ALL}') : embedding exists — skipping UMAP.")
else:
    print(f"All EPI ('{EMB_ALL}') : computing UMAP on {len(EPI)} epigenetic marks ...")
    run.compute_umap(markers=EPI, source_layer=ZS_OF['single'],
                     embedding_name=EMB_ALL, **UMAP_KW)

run.cluster_leiden(embedding_name=EMB_ALL, cluster_key=CLKEY_ALL,
                   resolution=RESOLUTION, seed=SEED)

# --- Set 2: 16 Epigenetic Marks (without H3K27me3 and H3K36me3) ---
if EMB_SUB in existing_emb:
    print(f"Sub EPI ('{EMB_SUB}') : embedding exists — skipping UMAP.")
else:
    print(f"Sub EPI ('{EMB_SUB}') : computing UMAP on {len(EPI_SUB)} epigenetic marks ...")
    run.compute_umap(markers=EPI_SUB, source_layer=ZS_OF['single'],
                     embedding_name=EMB_SUB, **UMAP_KW)

run.cluster_leiden(embedding_name=EMB_SUB, cluster_key=CLKEY_SUB,
                   resolution=RESOLUTION, seed=SEED)

adata = run.read_adata()
print(f"\nCombined Set 1 (All EPI, 18 marks)                    -> {adata.obs[CLKEY_ALL].nunique()} Leiden clusters ({CLKEY_ALL})")
print(f"Combined Set 2 (Sub EPI w/o H3K27me3 & H3K36me3, 16 marks) -> {adata.obs[CLKEY_SUB].nunique()} Leiden clusters ({CLKEY_SUB})")


In [ ]:
adata = run.read_adata()
adata.obs["sample_id"] = adata.obs["sample_id"].astype("category")

if "sample_id_colors" not in adata.uns:
    sc.pl.embedding(adata, basis=EMB_ALL, color="sample_id", palette="Set2", show=False)

categories = list(adata.obs["sample_id"].cat.categories)
sample_colors = dict(zip(categories, adata.uns["sample_id_colors"]))

adata.obs[CLKEY_ALL] = adata.obs[CLKEY_ALL].astype(str).astype("category")
adata.obs[CLKEY_SUB] = adata.obs[CLKEY_SUB].astype(str).astype("category")

# --- 1. Joint UMAP Clusters & Overview ---
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
sc.pl.embedding(
    adata, basis=EMB_ALL, color=CLKEY_ALL, ax=axes[0], show=False,
    legend_loc="on data", legend_fontsize=9, legend_fontoutline=2,
    palette="tab20", size=3, frameon=False, use_raw=False,
    title=f"Combined Set 1 (All 18 EPI) — {adata.obs[CLKEY_ALL].nunique()} Leiden Clusters",
)
sc.pl.embedding(
    adata, basis=EMB_SUB, color=CLKEY_SUB, ax=axes[1], show=False,
    legend_loc="on data", legend_fontsize=9, legend_fontoutline=2,
    palette="tab20", size=3, frameon=False, use_raw=False,
    title=f"Combined Set 2 (Sub EPI) — {adata.obs[CLKEY_SUB].nunique()} Leiden Clusters",
)
plt.tight_layout()
plt.savefig(PLOTS / f"{LINE}_norm_umap_comparison_clusters_combined.png", dpi=180, bbox_inches="tight")
plt.show()

# --- 2. Multi-Pane Sample Distribution Plot (One Pane per Sample) ---
n_samples = len(categories)
ncols = min(4, n_samples)
nrows = int(np.ceil(n_samples / ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 4.2, nrows * 4.0), squeeze=False)
axes_flat = axes.ravel()
all_coords = adata.obsm[EMB_ALL]

for idx, sample_name in enumerate(categories):
    ax = axes_flat[idx]
    ax.scatter(all_coords[:, 0], all_coords[:, 1], c="lightgrey", s=1, alpha=0.3)
    
    sub_mask = adata.obs["sample_id"] == sample_name
    sub_coords = all_coords[sub_mask]
    ax.scatter(sub_coords[:, 0], sub_coords[:, 1], c=[sample_colors[sample_name]], s=3, alpha=0.8)
    ax.set_title(f"Sample: {sample_name} ({sub_mask.sum():,} cells)", fontsize=11, fontweight="bold")
    ax.axis("off")

for ax in axes_flat[n_samples:]:
    ax.axis("off")

fig.suptitle("Combined Set 1 (All 18 EPI) — Samples across Panes", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig(PLOTS / f"{LINE}_norm_umap_samples_panes.png", dpi=180, bbox_inches="tight")
plt.show()


In [ ]:
ALL_MARKERS_SORTED = sorted(EPI + OTHER, key=lambda s: s.lower())
NONEPI = [m for m in OTHER if m in ALL_MARKERS]

# --- 1. Marker Grids for Combined Dataset ---
def marker_grid(emb_key, markers, title_prefix, ncols=5, fname=None):
    nrows = int(np.ceil(len(markers) / ncols))
    fig, axes_arr = plt.subplots(nrows, ncols, figsize=(ncols * 3.8, nrows * 3.6))
    axes_flat = np.atleast_1d(axes_arr).ravel()

    for idx, mname in enumerate(markers):
        ax = axes_flat[idx]
        sc.pl.embedding(
            adata, basis=emb_key, color=mname, layer=ZS_OF['single'],
            cmap="seismic", vcenter=0, vmin="p1", vmax="p99",
            ax=ax, size=4, frameon=False, use_raw=False,
            colorbar_loc="right", show=False
        )
        ax.set_title(mname, fontsize=13, fontweight="bold", pad=6)

    for ax in axes_flat[len(markers):]:
        ax.axis("off")

    fig.suptitle(f"{LINE_DISP} — Combined Dataset — {title_prefix} — All Markers (Z-Scored, Seismic)",
                 fontsize=18, fontweight="bold", y=1.02)
    plt.tight_layout()
    if fname:
        fig.savefig(PLOTS / fname, dpi=200, bbox_inches="tight")
    plt.show()

marker_grid(EMB_ALL, ALL_MARKERS_SORTED, title_prefix="Set 1 (All 18 EPI)", fname=f"{LINE}_umap_combined_all_epi_allmarkers.png")
marker_grid(EMB_SUB, ALL_MARKERS_SORTED, title_prefix="Set 2 (Sub EPI w/o H3K27me3 & H3K36me3)", fname=f"{LINE}_umap_combined_sub_epi_allmarkers.png")

# --- 2. Cluster-Based Heatmaps (Epigenetic & Non-Epigenetic for Both Clusterings) ---
# Set 1 (All EPI) — Epigenetic Modifications across Leiden Clusters
plt.figure(figsize=(10, 6))
run.plot_heatmap(EPI, CLKEY_ALL, layer=ZS_OF['single'], center=0)
plt.title("Combined Set 1 (All EPI) — Epigenetic Modifications across Leiden Clusters")
plt.savefig(PLOTS / f"{LINE}_heatmap_combined_set1_epigenetic.png", dpi=180, bbox_inches="tight")
plt.show()

# Set 1 (All EPI) — Non-Epigenetic Modifications across Leiden Clusters
plt.figure(figsize=(10, 6))
run.plot_heatmap(NONEPI, CLKEY_ALL, layer=ZS_OF['single'], center=0)
plt.title("Combined Set 1 (All EPI) — Non-Epigenetic Modifications across Leiden Clusters")
plt.savefig(PLOTS / f"{LINE}_heatmap_combined_set1_nonepigenetic.png", dpi=180, bbox_inches="tight")
plt.show()

# Set 2 (Sub EPI) — Epigenetic Modifications across Leiden Clusters
plt.figure(figsize=(10, 6))
run.plot_heatmap(EPI_SUB, CLKEY_SUB, layer=ZS_OF['single'], center=0)
plt.title("Combined Set 2 (Sub EPI w/o H3K27me3 & H3K36me3) — Epigenetic Modifications across Leiden Clusters")
plt.savefig(PLOTS / f"{LINE}_heatmap_combined_set2_epigenetic.png", dpi=180, bbox_inches="tight")
plt.show()

# Set 2 (Sub EPI) — Non-Epigenetic Modifications across Leiden Clusters
plt.figure(figsize=(10, 6))
run.plot_heatmap(NONEPI, CLKEY_SUB, layer=ZS_OF['single'], center=0)
plt.title("Combined Set 2 (Sub EPI w/o H3K27me3 & H3K36me3) — Non-Epigenetic Modifications across Leiden Clusters")
plt.savefig(PLOTS / f"{LINE}_heatmap_combined_set2_nonepigenetic.png", dpi=180, bbox_inches="tight")
plt.show()

# --- 3. Sample ID-Based Heatmaps (Epigenetic & Non-Epigenetic across Samples) ---
plt.figure(figsize=(10, 5))
run.plot_heatmap(EPI, "sample_id", layer=ZS_OF['single'], center=0)
plt.title("Combined Dataset — Epigenetic Modifications across Samples")
plt.savefig(PLOTS / f"{LINE}_heatmap_combined_sample_id_epigenetic.png", dpi=180, bbox_inches="tight")
plt.show()

plt.figure(figsize=(10, 5))
run.plot_heatmap(NONEPI, "sample_id", layer=ZS_OF['single'], center=0)
plt.title("Combined Dataset — Non-Epigenetic Modifications across Samples")
plt.savefig(PLOTS / f"{LINE}_heatmap_combined_sample_id_nonepigenetic.png", dpi=180, bbox_inches="tight")
plt.show()


In [ ]:
run.plot_cluster_composition('sample_id','cl_single')
plt.savefig('Plots/Cluster_Composition.png',dpi=200,bbox_inches='tight')

In [ ]:
adata = run.read_adata()
nh = uns_history(adata, "normalization")
prov = pd.DataFrame([{
    "method": h["method"], "gamma_mode": h["gamma_mode"],
    "tech_factor_kind": h["tech_factor_kind"], "corrected_layer": h["corrected_layer"],
    "module_version": h["module_version"], "entry_point": h["entry_point"],
    "arcsinh_cofactor": h["arcsinh_cofactor"], "n_cells": h["n_after"],
} for h in nh])
print(f"project : {PROJECT_PATH}")
print(f"run     : {RUN_ID}   ({adata.n_obs:,} cells x {adata.n_vars} markers)")
print(f"layers  : {sorted(adata.layers.keys())}")
print(f"plots   : {PLOTS}")
prov
